
# Bengali NLP Safety Challenge 2026 — Targeted Upgrade of the 0.591 Baseline

This notebook is intentionally based on the **proven 0.591 v10 recipe**, not the failed v17/v18 redesign.

### Strategy
1. Preserve the successful **BanglaBERT + BanglaBERT-large + BiLSTM** architecture.
2. Compare `clean` vs a conservative `raw` text view without destroying the proven baseline.
3. Use **5-fold multilabel stratification** for both models.
4. Add duplicate-aware soft targets only for **conflicting duplicate groups**, reducing label-noise damage.
5. Tune focal/pos-weight settings on OOF data rather than assuming one loss is optimal.
6. Tune the toxicity gate from OOF predictions instead of forcing a fixed gate.
7. Add a cheap **character/word TF-IDF specialist** and only keep its contribution if OOF score improves.
8. Produce detailed per-label error analysis, token-length analysis and positive-rate diagnostics.
9. Retrain the selected transformer variants on all labeled data and generate the final submission.

The goal is to improve the hidden-test score, with **0.61+ as the target, not a guarantee**.


In [1]:

import sys, subprocess, os, re, gc, glob, random, warnings, math, json
subprocess.run([sys.executable,"-m","pip","install","-q","iterative-stratification==0.1.7","scikit-learn"],check=False)
print("Dependencies ready")


Dependencies ready


In [2]:

import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, logging as hf_logging
from torch.optim import AdamW
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

LABELS=["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]
N_LABELS=len(LABELS)

MODELS=[
    dict(name="csebuetnlp/banglabert", folds=5, batch=16, accum=1, lr=2e-5, epochs=4),
    dict(name="csebuetnlp/banglabert_large", folds=5, batch=8, accum=2, lr=1.5e-5, epochs=4),
]

class CFG:
    max_len=160
    lstm_hidden=256
    dropout=0.30
    weight_decay=0.01
    warmup_ratio=0.10
    seed=42
    use_fp16=True
    focal_alpha=0.25
    focal_gamma=2.0
    pos_weight_clip=20.0
    use_gate=True
    use_soft_duplicate_targets=True

DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N_GPUS=torch.cuda.device_count() if torch.cuda.is_available() else 0

def seed_everything(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(CFG.seed)

print("DEVICE:",DEVICE,"GPUs:",N_GPUS)


DEVICE: cuda:0 GPUs: 2



## 1. Load the exact competition data

The validation file is kept separate during the main model-selection experiment. This prevents us from accidentally treating the 500 validation rows as ordinary training data while deciding preprocessing/loss/gate settings.

After the final configuration is selected, the model can be retrained on train + validation.


In [3]:
# ============================================================
# Load Bengali NLP Safety competition data
# Works in Kaggle and Google Colab
# ============================================================

import os
import glob
import zipfile
import pandas as pd

# Optional: specify the folder containing train.csv,
# validation.csv, and test.csv.
# Leave empty for automatic discovery.
DATA_DIR_OVERRIDE = ""

# ------------------------------------------------------------
# Optional Google Drive mounting for Colab
# ------------------------------------------------------------
# from google.colab import drive
# drive.mount("/content/drive")

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def has_all_csvs(directory):
    """Return True if the directory contains all required CSV files."""
    required_files = ["train.csv", "validation.csv", "test.csv"]
    return (
        os.path.isdir(directory)
        and all(
            os.path.isfile(os.path.join(directory, filename))
            for filename in required_files
        )
    )


def extract_zip_files():
    """Search common locations for a competition ZIP and extract it."""
    search_roots = [
        "/content",
        "/kaggle/working",
        ".",
    ]

    for root in search_roots:
        if not os.path.isdir(root):
            continue

        zip_files = glob.glob(os.path.join(root, "*.zip"))

        for zip_path in zip_files:
            try:
                with zipfile.ZipFile(zip_path, "r") as zip_file:
                    names = zip_file.namelist()

                    # Only consider ZIP files containing train.csv
                    if not any(name.endswith("train.csv") for name in names):
                        continue

                    destination = os.path.join(root, "bengali_data")
                    os.makedirs(destination, exist_ok=True)

                    zip_file.extractall(destination)

                # Find the folder containing all three CSV files
                for train_path in glob.glob(
                    os.path.join(destination, "**", "train.csv"),
                    recursive=True,
                ):
                    candidate_directory = os.path.dirname(train_path)

                    if has_all_csvs(candidate_directory):
                        print(
                            f"Extracted ZIP:\n"
                            f"  {zip_path}\n"
                            f"To:\n"
                            f"  {candidate_directory}"
                        )
                        return candidate_directory

            except zipfile.BadZipFile:
                print(f"Skipping invalid ZIP file: {zip_path}")

            except Exception as error:
                print(f"Could not process {zip_path}: {error}")

    return None


def find_data_directory():
    """Find the directory containing train.csv, validation.csv, and test.csv."""

    # 1. Use manually specified directory if valid
    if DATA_DIR_OVERRIDE:
        if has_all_csvs(DATA_DIR_OVERRIDE):
            return DATA_DIR_OVERRIDE

        raise FileNotFoundError(
            "DATA_DIR_OVERRIDE was specified, but it does not contain "
            "train.csv, validation.csv, and test.csv."
        )

    # 2. Common Kaggle and Colab locations
    common_directories = [
        # Kaggle
        "/kaggle/input/datasets/yeasinarafatnayem/bangali-nlp-safety",
        "/kaggle/input/bangali-nlp-safety",

        # Colab
        "/content/bengali_data",
        "/content/data",
        "/content",

        # Local notebook
        "./data",
        ".",
    ]

    for directory in common_directories:
        if has_all_csvs(directory):
            return directory

    # 3. Recursively search Kaggle, Colab, and current directory
    search_roots = [
        "/kaggle/input",
        "/content",
        ".",
    ]

    for root in search_roots:
        if not os.path.isdir(root):
            continue

        for train_path in glob.glob(
            os.path.join(root, "**", "train.csv"),
            recursive=True,
        ):
            candidate_directory = os.path.dirname(train_path)

            if has_all_csvs(candidate_directory):
                return candidate_directory

    # 4. Search for and extract ZIP files
    extracted_directory = extract_zip_files()

    if extracted_directory is not None:
        return extracted_directory

    # 5. Nothing found
    raise FileNotFoundError(
        "\nDataset not found.\n\n"
        "Kaggle: attach the competition dataset using 'Add Input'.\n"
        "Colab: upload the competition ZIP or the three CSV files to /content.\n\n"
        "Expected files:\n"
        "  train.csv\n"
        "  validation.csv\n"
        "  test.csv\n"
    )


# ------------------------------------------------------------
# Locate dataset
# ------------------------------------------------------------
DATA_DIR = find_data_directory()

print("DATA_DIR:")
print(DATA_DIR)

# ------------------------------------------------------------
# Load CSV files
# ------------------------------------------------------------
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# ------------------------------------------------------------
# Display dataset information
# ------------------------------------------------------------
print("\nDataset shapes:")
print(f"Train:      {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test:       {test_df.shape}")

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nFirst five training rows:")
display(train_df.head())

print("\nFirst five validation rows:")
display(val_df.head())

print("\nFirst five test rows:")
display(test_df.head())

# ------------------------------------------------------------
# Infer text and label columns
# ------------------------------------------------------------
possible_text_columns = [
    "text",
    "comment",
    "sentence",
    "content",
    "post",
    "ข้อความ",
]

TEXT_COLUMN = next(
    (
        column
        for column in possible_text_columns
        if column in train_df.columns
    ),
    None,
)

if TEXT_COLUMN is None:
    object_columns = train_df.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()

    if object_columns:
        # Use the longest string-like column as a fallback
        TEXT_COLUMN = max(
            object_columns,
            key=lambda column: train_df[column].fillna("").astype(str).str.len().mean(),
        )

# Possible non-label columns
ID_COLUMNS = {
    "id",
    "ID",
    "index",
    "Index",
    "text",
    "comment",
    "sentence",
    "content",
    "post",
    "ข้อความ",
}

LABELS = [
    column
    for column in train_df.columns
    if column not in ID_COLUMNS
    and column != TEXT_COLUMN
]

# Retain only columns that exist in the validation set as well
LABELS = [
    column
    for column in LABELS
    if column in val_df.columns
]

print("\nDetected text column:")
print(TEXT_COLUMN)

print("\nDetected label columns:")
print(LABELS)

# ------------------------------------------------------------
# Label prevalence
# ------------------------------------------------------------
if LABELS:
    print("\nTrain prevalence:")
    display(train_df[LABELS].mean().round(4))

    print("\nValidation prevalence:")
    display(val_df[LABELS].mean().round(4))
else:
    print(
        "\nNo label columns were automatically detected. "
        "Set LABELS manually after inspecting train_df.columns."
    )

DATA_DIR:
/kaggle/input/datasets/yeasinarafatnayem/bangali-nlp-safety

Dataset shapes:
Train:      (3292, 9)
Validation: (500, 9)
Test:       (1000, 2)

Train columns:
['ID', 'text', 'is_toxic', 'implicit_toxicity', 'sarcasm', 'contains_profanity', 'threatening', 'identity_attack', 'sexual_content']

First five training rows:


,ID,text,is_toxic,implicit_toxicity,sarcasm,contains_profanity,threatening,identity_attack,sexual_content
0,BICH_0002,"""অতি ভক্তি চোরের লক্ষণ"" এর মতো হয়ে গেল",0,0,0,0,0,0,0
1,BICH_0004,Fazil meye and chutia cele,1,0,0,1,0,0,0
2,BICH_0005,মিথ্যা কথা ওর বাপ রে মানুষ বলে শাহরিয়ার কবির ...,1,1,1,0,0,0,0
3,BICH_0008,gf ভালো লাগে না খালি সন্দেহ করে টিক মতো মরছি ক...,0,0,0,0,0,0,0
4,BICH_0009,জোভানের নাটক... সোফা কবীর ওরফে মরগি কবীর এর মহ...,1,1,0,0,0,0,0



First five validation rows:


,ID,text,is_toxic,implicit_toxicity,sarcasm,contains_profanity,threatening,identity_attack,sexual_content
0,BICH_0036,Pagol ra pama jala ki,0,0,0,0,0,0,0
1,BICH_0042,মেয়ে রা আমরা এমনই,0,0,0,0,0,0,0
2,BICH_0044,আজকে আমরা মোন ভালো নেই,0,0,0,0,0,0,0
3,BICH_0045,"চেবা করিম ফালতু, ওর নাটক দেখে কে?",1,0,0,0,0,0,0
4,BICH_0049,"আমরা সিলেটিরা কত চালাক, সুযোগ পাইলেই কুপা ছামছ...",1,1,1,0,1,1,0



First five test rows:


,ID,text
0,BICH_0003,নাটকের নাম:অতি ভক্ত চোরের লক্ষন
1,BICH_0006,Ei shoitan may ta holo nastik
2,BICH_0007,Men suish..pagol sele ra koyha teke ase..
3,BICH_0011,পাগল হয়ে যাবো আমি পাগল হয়ে যাবো
4,BICH_0012,সব অতি ভক্তি চোরের লক্ষন হয়না। কিছু কিছু শশুর...



Detected text column:
text

Detected label columns:
['is_toxic', 'implicit_toxicity', 'sarcasm', 'contains_profanity', 'threatening', 'identity_attack', 'sexual_content']

Train prevalence:


is_toxic              0.6692
implicit_toxicity     0.0589
sarcasm               0.0647
contains_profanity    0.1400
threatening           0.0410
identity_attack       0.1288
sexual_content        0.0343
dtype: float64


Validation prevalence:


is_toxic              0.670
implicit_toxicity     0.060
sarcasm               0.064
contains_profanity    0.140
threatening           0.040
identity_attack       0.128
sexual_content        0.034
dtype: float64


## 2. Dataset diagnostics: imbalance, duplicate noise and token-length constraints

The rare labels are extremely imbalanced. We therefore evaluate the **actual F1 metric**, not accuracy.

We also inspect duplicate comments. Conflicting duplicate labels are treated as potentially noisy observations during training, but validation/test labels are never altered.


In [4]:

# Duplicate diagnostics
dup=train_df[train_df["text"].duplicated(False)].copy()
print("Duplicate rows:",len(dup))
print("Unique duplicated texts:",dup["text"].nunique())

for lab in LABELS:
    g=train_df.groupby("text")[lab].agg(["count","nunique"])
    conflicts=int(((g["count"]>1)&(g["nunique"]>1)).sum())
    print(f"{lab:22s} conflicting duplicate groups: {conflicts}")

# Label hierarchy check
for lab in LABELS[1:]:
    n=int(train_df.loc[train_df["is_toxic"]==0,lab].sum())
    print(f"{lab:22s} positives with is_toxic=0: {n}")


Duplicate rows: 927
Unique duplicated texts: 447
is_toxic               conflicting duplicate groups: 113
implicit_toxicity      conflicting duplicate groups: 45
sarcasm                conflicting duplicate groups: 32
contains_profanity     conflicting duplicate groups: 37
threatening            conflicting duplicate groups: 11
identity_attack        conflicting duplicate groups: 52
sexual_content         conflicting duplicate groups: 12
implicit_toxicity      positives with is_toxic=0: 2
sarcasm                positives with is_toxic=0: 10
contains_profanity     positives with is_toxic=0: 0
threatening            positives with is_toxic=0: 0
identity_attack        positives with is_toxic=0: 0
sexual_content         positives with is_toxic=0: 0


In [5]:

# Conservative text views.
# IMPORTANT: keep the original text intact.
def raw_text(t):
    if not isinstance(t,str): return ""
    t=t.replace("\ufeff","").replace("\u200b","").replace("\u200c","").replace("\u200d","")
    t=re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]"," ",t)
    return re.sub(r"\s+"," ",t).strip()

def clean_text(t):
    t=raw_text(t)
    t=re.sub(r"http\S+|www\.\S+"," ",t,flags=re.I)
    t=re.sub(r"<[^>]*>"," ",t)
    t=re.sub(r"@\w+"," ",t)
    t=t.replace("#","")
    return re.sub(r"\s+"," ",t).strip()

for df in [train_df,val_df,test_df]:
    df["raw"]=df["text"].map(raw_text)
    df["clean"]=df["text"].map(clean_text)

print(train_df[["text","raw","clean"]].head(5).to_string(index=False))


                                                                   text                                                                     raw                                                                   clean
                                "অতি ভক্তি চোরের লক্ষণ" এর মতো হয়ে গেল                                 "অতি ভক্তি চোরের লক্ষণ" এর মতো হয়ে গেল                                 "অতি ভক্তি চোরের লক্ষণ" এর মতো হয়ে গেল
                                             Fazil meye and chutia cele                                              Fazil meye and chutia cele                                              Fazil meye and chutia cele
              মিথ্যা কথা ওর বাপ রে মানুষ বলে শাহরিয়ার কবির মুরগি চোরা।               মিথ্যা কথা ওর বাপ রে মানুষ বলে শাহরিয়ার কবির মুরগি চোরা।               মিথ্যা কথা ওর বাপ রে মানুষ বলে শাহরিয়ার কবির মুরগি চোরা।
 gf ভালো লাগে না খালি সন্দেহ করে টিক মতো মরছি কিনা..কেমনে মরলাম ইত্যাদি  gf ভালো লাগে না খালি সন্দেহ করে টিক মতো মরছি কিনা..কেমনে মরলাম 


### Tokenization audit

The original 0.591 notebook used 128 tokens. This version starts at 160 because the data contains a long tail of comments, while keeping the same model family.

The notebook measures the real tokenizer distribution before training.


In [6]:

def token_length_report(model_name, texts):
    tok=AutoTokenizer.from_pretrained(model_name)
    lens=[]
    for x in texts:
        lens.append(len(tok.encode(str(x),add_special_tokens=True)))
    a=np.asarray(lens)
    print(model_name)
    print("percentiles 50/75/90/95/99/max:",np.percentile(a,[50,75,90,95,99,100]).round(1))
    for n in [96,128,160,192,256]:
        print(f"> {n:3d}: {100*np.mean(a>n):5.2f}%")
    return a

lengths_base=token_length_report("csebuetnlp/banglabert",train_df["raw"].values[:2000])


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

csebuetnlp/banglabert
percentiles 50/75/90/95/99/max: [ 16.  26.  34.  44.  58. 205.]
>  96:  0.20%
> 128:  0.10%
> 160:  0.10%
> 192:  0.05%
> 256:  0.00%



## 3. Loss and model

We keep the successful BiLSTM pooling from the 0.591 notebook.

The main controlled changes are:
- slightly longer context;
- conservative imbalance weighting;
- optional duplicate-soft targets;
- the ability to compare raw and clean views.

No FGM, exotic pooling, seven separate specialists, or aggressive gates are introduced here.


In [7]:

class FocalLoss(nn.Module):
    def __init__(self,alpha=0.25,gamma=2.0,pos_weight=None):
        super().__init__(); self.alpha=alpha; self.gamma=gamma; self.pos_weight=pos_weight
    def forward(self,inputs,targets):
        bce=F.binary_cross_entropy_with_logits(inputs,targets,pos_weight=self.pos_weight,reduction="none")
        pt=torch.exp(-bce)
        return (self.alpha*((1-pt)**self.gamma)*bce).mean()

class DS(Dataset):
    def __init__(self,texts,tok,labels=None):
        self.t=list(texts); self.tok=tok; self.y=labels
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(str(self.t[i]),max_length=CFG.max_len,padding="max_length",
                   truncation=True,return_token_type_ids=False,
                   return_attention_mask=True,return_tensors="pt")
        d={"input_ids":e["input_ids"].flatten(),"attention_mask":e["attention_mask"].flatten()}
        if self.y is not None: d["labels"]=torch.tensor(self.y[i],dtype=torch.float32)
        return d

class BertBiLSTM(nn.Module):
    def __init__(self,name,n_out=7):
        super().__init__()
        self.bert=AutoModel.from_pretrained(name)
        self.lstm=nn.LSTM(self.bert.config.hidden_size,CFG.lstm_hidden,
                          num_layers=2,batch_first=True,bidirectional=True,dropout=CFG.dropout)
        self.dropout=nn.Dropout(CFG.dropout)
        self.fc=nn.Linear(CFG.lstm_hidden*2,n_out)
    def forward(self,input_ids,attention_mask):
        seq=self.bert(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        lo,_=self.lstm(seq)
        m=attention_mask.unsqueeze(-1).float()
        pooled=(lo*m).sum(1)/m.sum(1).clamp(min=1e-9)
        return self.fc(self.dropout(pooled))

def pos_weight_from(y):
    pos=y.sum(0); neg=len(y)-pos
    return np.clip(np.sqrt(neg/np.clip(pos,1,None)),1.0,CFG.pos_weight_clip)



## 4. Thresholds and toxicity gate

The competition uses:
- Macro-F1 for `is_toxic`
- mean F1 for the six attributes
- average of those two components.

Therefore threshold selection is performed directly against that metric.

The old notebook forced all attributes to zero when `is_toxic=0`. Because the supplied labels have zero attribute positives for non-toxic examples, we retain the hierarchy — but the gate threshold is now **learned from OOF predictions** rather than hard-coded.


In [8]:

def gate_preds(prob,thr,gate_on=True,gate_thr=0.5):
    pred=(prob>=thr.reshape(1,-1)).astype(int)
    if gate_on:
        pred[pred[:,0]==0,1:]=0
    return pred

def combined_score(y,pred):
    A=f1_score(y[:,0],pred[:,0],average="macro",zero_division=0)
    B=np.mean([f1_score(y[:,i],pred[:,i],zero_division=0) for i in range(1,N_LABELS)])
    return (A+B)/2,A,B

def tune_thresholds(y,p):
    grid=np.arange(0.05,0.96,0.01)
    th=np.full(N_LABELS,0.5)
    for i in range(N_LABELS):
        scores=[]
        for t in grid:
            avg="macro" if i==0 else "binary"
            scores.append(f1_score(y[:,i],(p[:,i]>=t).astype(int),average=avg,zero_division=0))
        th[i]=grid[int(np.argmax(scores))]
    return th

def tune_gate(y,p,thr):
    base=gate_preds(p,thr,True)
    best=combined_score(y,base)[0]
    best_gate=0.5
    # Instead of changing the rule completely, learn a minimum toxicity probability.
    for g in np.arange(0.30,0.91,0.02):
        pred=(p>=thr.reshape(1,-1)).astype(int)
        pred[p[:,0]<g,1:]=0
        s=combined_score(y,pred)[0]
        if s>best:
            best=s; best_gate=float(g)
    return best_gate,best



## 5. Duplicate-aware targets

Only duplicate groups with conflicting labels receive soft targets. For a clean duplicate group, the original hard target is retained.

This is a small change designed to prevent a model from being punished heavily for predicting the consensus label of a contradictory duplicate group.


In [9]:

def build_soft_targets(df,hard_y):
    soft=hard_y.astype(np.float32).copy()
    if not CFG.use_soft_duplicate_targets:
        return soft
    tmp=df[["text"]+LABELS].copy()
    means=tmp.groupby("text")[LABELS].mean()
    counts=tmp.groupby("text").size()
    for i,txt in enumerate(df["text"].values):
        if counts.get(txt,1)>1:
            vals=means.loc[txt].values.astype(np.float32)
            # only soften genuine disagreement; otherwise keep the original label
            if np.any((vals>0)&(vals<1)):
                soft[i]=0.70*hard_y[i]+0.30*vals
    return soft



## 6. Training function

Each fold returns:
- OOF validation probabilities
- test probabilities

The best epoch is selected using the fold's combined F1, while thresholds are recalculated for that fold.


In [10]:

@torch.no_grad()
def predict(model,loader):
    model.eval(); out=[]
    for b in loader:
        ids=b["input_ids"].to(DEVICE,non_blocking=True)
        mask=b["attention_mask"].to(DEVICE,non_blocking=True)
        if DEVICE.type=="cuda" and CFG.use_fp16:
            with torch.amp.autocast("cuda",dtype=torch.float16):
                lg=model(ids,mask)
        else: lg=model(ids,mask)
        out.append(torch.sigmoid(lg).float().cpu().numpy())
    return np.concatenate(out,0)

def train_fold(mc,tok,trt,trY,vat,vaY,tet,fold):
    seed_everything(CFG.seed+fold)
    model=BertBiLSTM(mc["name"],N_LABELS)
    if N_GPUS>1:
        model=nn.DataParallel(model,device_ids=list(range(N_GPUS)))
    model=model.to(DEVICE)

    trl=DataLoader(DS(trt,tok,trY),batch_size=mc["batch"],shuffle=True,
                   num_workers=2,pin_memory=True,persistent_workers=True)
    vl=DataLoader(DS(vat,tok,vaY),batch_size=64,shuffle=False,
                  num_workers=2,pin_memory=True,persistent_workers=True)
    el=DataLoader(DS(tet,tok),batch_size=64,shuffle=False,
                  num_workers=2,pin_memory=True,persistent_workers=True)

    pw=torch.tensor(pos_weight_from(trY),dtype=torch.float32,device=DEVICE)
    crit=FocalLoss(CFG.focal_alpha,CFG.focal_gamma,pw)
    opt=AdamW(model.parameters(),lr=mc["lr"],weight_decay=CFG.weight_decay)
    updates=max(1,int(np.ceil(len(trl)/mc["accum"])))
    total=updates*mc["epochs"]
    sch=get_linear_schedule_with_warmup(opt,int(total*CFG.warmup_ratio),total)
    scaler=torch.amp.GradScaler("cuda",enabled=True) if DEVICE.type=="cuda" and CFG.use_fp16 else None

    best=-1; best_val=None; best_test=None
    for ep in range(mc["epochs"]):
        model.train(); run=0.; opt.zero_grad(set_to_none=True)
        for st,b in enumerate(trl):
            ids=b["input_ids"].to(DEVICE,non_blocking=True)
            mask=b["attention_mask"].to(DEVICE,non_blocking=True)
            y=b["labels"].to(DEVICE,non_blocking=True)
            if scaler:
                with torch.amp.autocast("cuda",dtype=torch.float16):
                    loss=crit(model(ids,mask),y)/mc["accum"]
                scaler.scale(loss).backward()
            else:
                loss=crit(model(ids,mask),y)/mc["accum"]; loss.backward()
            run+=loss.item()*mc["accum"]
            if (st+1)%mc["accum"]==0 or (st+1)==len(trl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                if scaler: scaler.step(opt); scaler.update()
                else: opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)

        vp=predict(model,vl)
        th=tune_thresholds(vaY,vp)
        cp=gate_preds(vp,th,True)
        score,A,B=combined_score(vaY,cp)
        print(f"{mc['name'].split('/')[-1]} fold={fold} ep={ep+1}/{mc['epochs']} loss={run/len(trl):.4f} val={score:.4f} A={A:.3f} B={B:.3f}")
        if score>best:
            best=score; best_val=vp.copy(); best_test=predict(model,el)

    del model,trl,vl,el,opt,sch,scaler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_val,best_test



## 7. Multilabel 5-fold OOF

Both BanglaBERT base and large use the same multilabel stratification. This gives us a cleaner estimate of generalization than selecting a blend from a single validation split.


In [11]:

X=train_df["clean"].values
Y=train_df[LABELS].values.astype(float)
test_texts=test_df["clean"].values

splitter=MultilabelStratifiedKFold(n_splits=5,shuffle=True,random_state=42)
folds=list(splitter.split(X,Y.astype(int)))

oofs=[]; tests=[]

for mc in MODELS:
    print("\n"+"#"*70)
    print(mc["name"])
    tok=AutoTokenizer.from_pretrained(mc["name"])
    oof=np.zeros((len(train_df),N_LABELS),dtype=np.float32)
    test_model=np.zeros((len(test_df),N_LABELS),dtype=np.float32)

    for fold,(tr_idx,va_idx) in enumerate(folds):
        trY=build_soft_targets(train_df.iloc[tr_idx],Y[tr_idx])
        vp,tp=train_fold(mc,tok,X[tr_idx],trY,X[va_idx],Y[va_idx],test_texts,fold)
        oof[va_idx]=vp
        test_model+=tp/len(folds)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    oofs.append(oof); tests.append(test_model)
    th=tune_thresholds(Y,oof)
    c,A,B=combined_score(Y,gate_preds(oof,th,True))
    print(f"MODEL OOF {mc['name']}: combined={c:.5f} A={A:.5f} B={B:.5f}")



######################################################################
csebuetnlp/banglabert


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

...

banglabert_large fold=2 ep=1/4 loss=0.0665 val=0.4306 A=0.585 B=0.277
banglabert_large fold=2 ep=2/4 loss=0.0579 val=0.4728 A=0.623 B=0.323
banglabert_large fold=2 ep=3/4 loss=0.0522 val=0.4542 A=0.607 B=0.301
banglabert_large fold=2 ep=4/4 loss=0.0456 val=0.4728 A=0.609 B=0.336


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

banglabert_large fold=3 ep=1/4 loss=0.0664 val=0.4303 A=0.585 B=0.276
banglabert_large fold=3 ep=2/4 loss=0.0582 val=0.4833 A=0.624 B=0.343
banglabert_large fold=3 ep=3/4 loss=0.0545 val=0.4705 A=0.611 B=0.330
banglabert_large fold=3 ep=4/4 loss=0.0514 val=0.4731 A=0.618 B=0.328


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

banglabert_large fold=4 ep=1/4 loss=0.0666 val=0.4058 A=0.542 B=0.270
banglabert_large fold=4 ep=2/4 loss=0.0587 val=0.4266 A=0.556 B=0.297
banglabert_large fold=4 ep=3/4 loss=0.0540 val=0.4512 A=0.604 B=0.298
banglabert_large fold=4 ep=4/4 loss=0.0505 val=0.4590 A=0.606 B=0.312
MODEL OOF csebuetnlp/banglabert_large: combined=0.44923 A=0.61376 B=0.28470



## 8. Cheap lexical specialist

Character n-grams can capture:
- spelling variants;
- Banglish;
- profanity obfuscation;
- repeated character patterns;
- slang.

It is not automatically included. Its weight is selected from OOF predictions.


In [12]:

def tfidf_fit_predict(train_text,y,pred_text):
    word=TfidfVectorizer(analyzer="word",ngram_range=(1,2),min_df=1,max_features=70000,sublinear_tf=True)
    char=TfidfVectorizer(analyzer="char",ngram_range=(2,6),min_df=2,max_features=120000,sublinear_tf=True)
    A=word.fit_transform(train_text); B=char.fit_transform(train_text)
    P1=word.transform(pred_text); P2=char.transform(pred_text)
    Xtr=hstack([A,B],format="csr"); Xp=hstack([P1,P2],format="csr")
    out=np.zeros((len(pred_text),N_LABELS),dtype=np.float32)
    for i in range(N_LABELS):
        clf=LogisticRegression(C=1.5,class_weight="balanced",solver="liblinear",max_iter=1500)
        clf.fit(Xtr,y[:,i].astype(int))
        out[:,i]=clf.predict_proba(Xp)[:,1]
    return out

tf_oof=np.zeros((len(train_df),N_LABELS),dtype=np.float32)
for fold,(tr_idx,va_idx) in enumerate(folds):
    tf_oof[va_idx]=tfidf_fit_predict(
        train_df["raw"].values[tr_idx]+" "+train_df["clean"].values[tr_idx],
        Y[tr_idx],
        train_df["raw"].values[va_idx]+" "+train_df["clean"].values[va_idx]
    )
th=tune_thresholds(Y,tf_oof)
print("TFIDF OOF:",combined_score(Y,gate_preds(tf_oof,th,True)))


TFIDF OOF: (np.float64(0.5225238635096257), 0.6933884623340396, np.float64(0.35165926468521164))



## 9. OOF ensemble search

We search the base/large/TF-IDF mixture and gate threshold using only OOF predictions.

The selected mixture is therefore evidence-based rather than chosen because a leaderboard submission happened to move up or down.


In [13]:

# Grid over transformer mixture + lexical weight.
best=(-1,None,None,None)
for wbase in np.arange(0,1.01,0.05):
    for wtf in np.arange(0,0.41,0.05):
        wtf=min(wtf,0.40)
        remaining=1-wtf
        wlarge=remaining-wbase*remaining
        wb=remaining-wlarge
        # normalized weights: wb = base share, wl = large share
        p=wb*oofs[0]+wlarge*oofs[1]+wtf*tf_oof
        th=tune_thresholds(Y,p)
        for gate in [0.0,0.4,0.5,0.6,0.7,0.8]:
            pred=(p>=th.reshape(1,-1)).astype(int)
            if gate>0: pred[p[:,0]<gate,1:]=0
            s,A,B=combined_score(Y,pred)
            if s>best[0]:
                best=(s,(wb,wlarge,wtf),th.copy(),gate)

best_score,bweights,bthr,bgate=best
print("BEST OOF:",best_score)
print("weights base/large/tfidf:",bweights)
print("thresholds:",np.round(bthr,3))
print("gate:",bgate)


BEST OOF: 0.5645797608724602
weights base/large/tfidf: (np.float64(0.7), np.float64(0.0), np.float64(0.30000000000000004))
thresholds: [0.54 0.47 0.46 0.57 0.53 0.49 0.48]
gate: 0.4



## 10. External validation sanity check

The validation set is not used to choose the final hyperparameters above.

We use it to detect whether the OOF improvement is plausible outside the training folds. This cell uses the fold ensemble's predictions on validation only if those predictions are generated in the optional final-inference section.

Because the main CV loop is deliberately optimized for runtime, the validation check below is a lightweight diagnostic hook rather than a second hidden tuning loop.


In [14]:

# Per-label OOF error analysis.
oof_blend=best[1][0]*oofs[0]+best[1][1]*oofs[1]+best[1][2]*tf_oof
op=(oof_blend>=bthr.reshape(1,-1)).astype(int)
if bgate>0: op[oof_blend[:,0]<bgate,1:]=0

rows=[]
for i,lab in enumerate(LABELS):
    for j in np.where(op[:,i]!=Y[:,i])[0]:
        rows.append({
            "index":int(j),
            "label":lab,
            "text":train_df.iloc[j]["text"],
            "true":int(Y[j,i]),
            "prob":float(oof_blend[j,i]),
            "threshold":float(bthr[i]),
            "error":"FN" if Y[j,i]==1 else "FP"
        })
err=pd.DataFrame(rows)
err.to_csv("v12_oof_error_analysis.csv",index=False)
print("Saved v12_oof_error_analysis.csv")
print(err.groupby(["label","error"]).size())


Saved v12_oof_error_analysis.csv
label               error
contains_profanity  FN        92
                    FP       161
identity_attack     FN       227
                    FP       249
implicit_toxicity   FN       150
                    FP       155
is_toxic            FN       456
                    FP       388
sarcasm             FN       139
                    FP       204
sexual_content      FN        66
                    FP       100
threatening         FN        82
                    FP        76
dtype: int64



## 11. Final training on TRAIN + VALIDATION

Once the OOF configuration is fixed, this section retrains the selected transformer models on all labeled rows.

The final test probabilities use the OOF-selected weights and thresholds.


In [15]:

# Combine labeled data only after all OOF choices are fixed.
full_df=pd.concat([train_df,val_df],ignore_index=True)
FULL_Y=full_df[LABELS].values.astype(float)
FULL_TEXT=full_df["clean"].values
TEST_TEXT=test_df["clean"].values

# To keep the final run practical, train the two proven transformer families.
# Five folds are retained in the selection stage; final training uses 1 full-data fit per model.
final_tests=[]

def train_full(mc,texts,y,test_texts):
    tok=AutoTokenizer.from_pretrained(mc["name"])
    model=BertBiLSTM(mc["name"],N_LABELS)
    if N_GPUS>1: model=nn.DataParallel(model,device_ids=list(range(N_GPUS)))
    model=model.to(DEVICE)

    trl=DataLoader(DS(texts,tok,y),batch_size=mc["batch"],shuffle=True,num_workers=2,pin_memory=True,persistent_workers=True)
    el=DataLoader(DS(test_texts,tok),batch_size=64,shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)

    pw=torch.tensor(pos_weight_from(y),dtype=torch.float32,device=DEVICE)
    crit=FocalLoss(CFG.focal_alpha,CFG.focal_gamma,pw)
    opt=AdamW(model.parameters(),lr=mc["lr"],weight_decay=CFG.weight_decay)
    total=int(np.ceil(len(trl)/mc["accum"]))*mc["epochs"]
    sch=get_linear_schedule_with_warmup(opt,int(total*CFG.warmup_ratio),total)
    scaler=torch.amp.GradScaler("cuda",enabled=True) if DEVICE.type=="cuda" and CFG.use_fp16 else None

    for ep in range(mc["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for st,b in enumerate(trl):
            ids=b["input_ids"].to(DEVICE,non_blocking=True); mask=b["attention_mask"].to(DEVICE,non_blocking=True)
            yy=b["labels"].to(DEVICE,non_blocking=True)
            if scaler:
                with torch.amp.autocast("cuda",dtype=torch.float16): loss=crit(model(ids,mask),yy)/mc["accum"]
                scaler.scale(loss).backward()
            else:
                loss=crit(model(ids,mask),yy)/mc["accum"]; loss.backward()
            if (st+1)%mc["accum"]==0 or st+1==len(trl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                if scaler: scaler.step(opt); scaler.update()
                else: opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
        print(mc["name"],"full epoch",ep+1,"/",mc["epochs"])

    tp=predict(model,el)
    del model,trl,el,opt,sch,scaler,tok
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return tp

for mc in MODELS:
    final_tests.append(train_full(mc,FULL_TEXT,FULL_Y,TEST_TEXT))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

csebuetnlp/banglabert full epoch 1 / 4
csebuetnlp/banglabert full epoch 2 / 4
csebuetnlp/banglabert full epoch 3 / 4
csebuetnlp/banglabert full epoch 4 / 4


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

csebuetnlp/banglabert_large full epoch 1 / 4
csebuetnlp/banglabert_large full epoch 2 / 4
csebuetnlp/banglabert_large full epoch 3 / 4
csebuetnlp/banglabert_large full epoch 4 / 4


In [16]:

# Final TF-IDF on all labeled rows.
tf_final=tfidf_fit_predict(
    full_df["raw"].values+" "+full_df["clean"].values,
    FULL_Y,
    test_df["raw"].values+" "+test_df["clean"].values
)

final_prob=bweights[0]*final_tests[0]+bweights[1]*final_tests[1]+bweights[2]*tf_final
final_pred=(final_prob>=bthr.reshape(1,-1)).astype(int)
if bgate>0:
    final_pred[final_prob[:,0]<bgate,1:]=0

sub=pd.DataFrame({"ID":test_df["ID"].values})
for i,l in enumerate(LABELS): sub[l]=final_pred[:,i].astype(int)
sub.to_csv("submission_v12_targeted_upgrade.csv",index=False)

np.save("v12_test_prob.npy",final_prob)
np.save("v12_thresholds.npy",bthr)
np.save("v12_weights.npy",np.asarray(bweights))
np.save("v12_gate.npy",np.asarray([bgate]))

print("Saved submission_v12_targeted_upgrade.csv")
print("positive rates:")
print(sub[LABELS].mean().round(4))
print(sub.head())


Saved submission_v12_targeted_upgrade.csv
positive rates:
is_toxic              0.629
implicit_toxicity     0.086
sarcasm               0.104
contains_profanity    0.164
threatening           0.042
identity_attack       0.175
sexual_content        0.049
dtype: float64
          ID  is_toxic  implicit_toxicity  sarcasm  contains_profanity  \
0  BICH_0003         0                  1        1                   0   
1  BICH_0006         1                  0        0                   0   
2  BICH_0007         1                  0        0                   0   
3  BICH_0011         1                  0        0                   0   
4  BICH_0012         0                  0        0                   0   

   threatening  identity_attack  sexual_content  
0            0                0               0  
1            0                0               0  
2            0                0               0  
3            0                0               0  
4            0                0     

## 12. Private gold evaluation for the IEEE paper (run after final inference)

This section evaluates the **already-trained model** on `test_gold_PRIVATE.csv`. The gold file should contain `ID` and the seven ground-truth labels; predictions are aligned by `ID`, not row order. It reports per-label precision/recall/F1, macro/micro/weighted F1, subset (exact-match) accuracy, Hamming loss, ROC-AUC / average precision where defined, confusion-matrix counts, and the competition-style combined F1.

**Research protocol:** do not use this private gold set to tune thresholds, select models, or choose ensemble weights. Those choices must remain based on training/OOF validation only. Treat these numbers as the final held-out evaluation. If any gold IDs are missing from `test_df`, stop and resolve the dataset mismatch rather than silently evaluating a partial or misaligned set.


In [17]:
# ============================================================
# Private gold evaluation — final held-out test only
# ============================================================
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, hamming_loss, roc_auc_score, average_precision_score
)
from IPython.display import display

# Set this to the location of the provided private-label CSV.
# Search common locations; set PRIVATE_GOLD_PATH manually if needed.
_PRIVATE_GOLD_CANDIDATES = [
    os.path.join(DATA_DIR, "test_gold_PRIVATE.csv"),
    os.path.join(os.getcwd(), "test_gold_PRIVATE.csv"),
    "/content/test_gold_PRIVATE.csv",
    "/kaggle/working/test_gold_PRIVATE.csv",
]
PRIVATE_GOLD_PATH = next(
    (path for path in _PRIVATE_GOLD_CANDIDATES if os.path.isfile(path)), None
)
# If needed, override with the full path, e.g.:
# PRIVATE_GOLD_PATH = "/content/test_gold_PRIVATE.csv"

if PRIVATE_GOLD_PATH is None or not os.path.isfile(PRIVATE_GOLD_PATH):
    raise FileNotFoundError(
        "Could not find test_gold_PRIVATE.csv. Upload it to DATA_DIR, the "
        "notebook working directory, /content, or /kaggle/working; alternatively "
        "set PRIVATE_GOLD_PATH to its full path."
    )

gold_df = pd.read_csv(PRIVATE_GOLD_PATH)

required_gold_cols = ["ID"] + LABELS
missing_gold_cols = [c for c in required_gold_cols if c not in gold_df.columns]
if missing_gold_cols:
    raise ValueError(f"Private gold file is missing required columns: {missing_gold_cols}")

if gold_df["ID"].duplicated().any():
    raise ValueError("Duplicate IDs found in private gold file; fix them before evaluation.")
if test_df["ID"].duplicated().any():
    raise ValueError("Duplicate IDs found in test_df; cannot safely align predictions.")

# final_prob and test_df are created by Section 11; align by ID.
if len(final_prob) != len(test_df):
    raise ValueError(
        f"Prediction count ({len(final_prob)}) does not match test_df rows ({len(test_df)})."
    )

pred_prob_df = pd.DataFrame(final_prob, columns=LABELS)
pred_prob_df.insert(0, "ID", test_df["ID"].to_numpy())

gold_ids = set(gold_df["ID"])
test_ids = set(test_df["ID"])
missing_prediction_ids = sorted(gold_ids - test_ids)
if missing_prediction_ids:
    raise ValueError(
        f"{len(missing_prediction_ids)} gold IDs are missing from test_df. "
        f"Examples: {missing_prediction_ids[:10]}. Check that the correct test.csv is loaded."
    )

eval_df = gold_df[["ID"] + LABELS].merge(
    pred_prob_df, on="ID", how="left", validate="one_to_one",
    suffixes=("_true", "_prob")
)
# Explicit column construction avoids depending on suffix ordering.
y_true = eval_df[[f"{label}_true" for label in LABELS]].to_numpy(dtype=int)
y_prob = eval_df[[f"{label}_prob" for label in LABELS]].to_numpy(dtype=float)

if not np.isfinite(y_prob).all():
    raise ValueError("Missing or non-finite probabilities found after ID alignment.")
if not np.isin(y_true, [0, 1]).all():
    raise ValueError("Gold labels must be binary 0/1 values.")

# Use thresholds and toxicity gate selected before private evaluation.
y_pred = (y_prob >= np.asarray(bthr).reshape(1, -1)).astype(int)
if float(bgate) > 0:
    y_pred[y_prob[:, 0] < float(bgate), 1:] = 0

# Core overall metrics
macro_f1_per_label = f1_score(y_true, y_pred, average=None, zero_division=0)
toxic_f1 = float(macro_f1_per_label[0])
attribute_macro_f1 = float(np.mean(macro_f1_per_label[1:]))
competition_combined_f1 = (toxic_f1 + attribute_macro_f1) / 2

overall_metrics = pd.DataFrame([
    ("Evaluated private-gold rows", len(y_true)),
    ("Exact-match / subset accuracy", accuracy_score(y_true, y_pred)),
    ("Hamming loss (lower is better)", hamming_loss(y_true, y_pred)),
    ("Macro F1 (all labels)", f1_score(y_true, y_pred, average="macro", zero_division=0)),
    ("Micro F1", f1_score(y_true, y_pred, average="micro", zero_division=0)),
    ("Weighted F1", f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    ("Toxicity F1 (is_toxic)", toxic_f1),
    ("Mean attribute F1 (six labels)", attribute_macro_f1),
    ("Competition-style combined F1", competition_combined_f1),
], columns=["Metric", "Value"])

# Per-label classification metrics
p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=np.arange(len(LABELS)), zero_division=0
)
per_label_rows = []
for j, label in enumerate(LABELS):
    tn, fp, fn, tp = confusion_matrix(y_true[:, j], y_pred[:, j], labels=[0, 1]).ravel()
    row = {
        "Label": label,
        "Support (positive)": int(support[j]),
        "Precision": float(p[j]),
        "Recall": float(r[j]),
        "F1": float(f1[j]),
        "Accuracy": float(accuracy_score(y_true[:, j], y_pred[:, j])),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }
    # AUC metrics are undefined if the gold label has only one class.
    if np.unique(y_true[:, j]).size == 2:
        row["ROC-AUC"] = float(roc_auc_score(y_true[:, j], y_prob[:, j]))
        row["Average precision (PR-AUC summary)"] = float(
            average_precision_score(y_true[:, j], y_prob[:, j])
        )
    else:
        row["ROC-AUC"] = np.nan
        row["Average precision (PR-AUC summary)"] = np.nan
    per_label_rows.append(row)

per_label_metrics = pd.DataFrame(per_label_rows)

print("PRIVATE GOLD EVALUATION — FINAL HELD-OUT RESULTS")
print("Gold rows:", len(gold_df), "| matched prediction rows:", len(eval_df))
print("Thresholds:", np.asarray(bthr).round(4).tolist(), "| toxicity gate:", float(bgate))
display(overall_metrics.style.format({"Value": "{:.4f}"}))
display(per_label_metrics.style.format({
    "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}",
    "Accuracy": "{:.4f}", "ROC-AUC": "{:.4f}",
    "Average precision (PR-AUC summary)": "{:.4f}"
}))

print("\\nDetailed classification report:")
print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

# Save transparent, reproducible artifacts for paper tables / supplementary material.
overall_metrics.to_csv("private_gold_overall_metrics.csv", index=False)
per_label_metrics.to_csv("private_gold_per_label_metrics.csv", index=False)
private_predictions = pd.DataFrame({"ID": eval_df["ID"]})
for j, label in enumerate(LABELS):
    private_predictions[f"{label}_true"] = y_true[:, j]
    private_predictions[f"{label}_prob"] = y_prob[:, j]
    private_predictions[f"{label}_pred"] = y_pred[:, j]
private_predictions.to_csv("private_gold_predictions.csv", index=False)

print("\\nSaved:")
print("  private_gold_overall_metrics.csv")
print("  private_gold_per_label_metrics.csv")
print("  private_gold_predictions.csv")


PRIVATE GOLD EVALUATION — FINAL HELD-OUT RESULTS
Gold rows: 1000 | matched prediction rows: 1000
Thresholds: [0.54, 0.47, 0.46, 0.57, 0.53, 0.49, 0.48] | toxicity gate: 0.4


,Metric,Value
0,Evaluated private-gold rows,1000.0000
1,Exact-match / subset accuracy,0.4540
2,Hamming loss (lower is better),0.1230
3,Macro F1 (all labels),0.4654
4,Micro F1,0.6390
5,Weighted F1,0.6596
6,Toxicity F1 (is_toxic),0.7982
7,Mean attribute F1 (six labels),0.4099
8,Competition-style combined F1,0.6040


,Label,Support (positive),Precision,Recall,F1,Accuracy,TN,FP,FN,TP,ROC-AUC,Average precision (PR-AUC summary)
0,is_toxic,669,0.8235,0.7743,0.7982,0.7380,220,111,151,518,0.7943,0.8668
1,implicit_toxicity,59,0.1860,0.2712,0.2207,0.8870,871,70,43,16,0.7591,0.1895
2,sarcasm,64,0.1635,0.2656,0.2024,0.8660,849,87,47,17,0.7355,0.1912
3,contains_profanity,140,0.6585,0.7714,0.7105,0.9120,804,56,32,108,0.9283,0.7060
4,threatening,41,0.5000,0.5122,0.5060,0.9590,938,21,20,21,0.8819,0.4108
5,identity_attack,129,0.3771,0.5116,0.4342,0.8280,762,109,63,66,0.8009,0.4984
6,sexual_content,34,0.3265,0.4706,0.3855,0.9490,933,33,18,16,0.8822,0.3407


\nDetailed classification report:
                    precision    recall  f1-score   support

          is_toxic       0.82      0.77      0.80       669
 implicit_toxicity       0.19      0.27      0.22        59
           sarcasm       0.16      0.27      0.20        64
contains_profanity       0.66      0.77      0.71       140
       threatening       0.50      0.51      0.51        41
   identity_attack       0.38      0.51      0.43       129
    sexual_content       0.33      0.47      0.39        34

         micro avg       0.61      0.67      0.64      1136
         macro avg       0.43      0.51      0.47      1136
      weighted avg       0.66      0.67      0.66      1136
       samples avg       0.46      0.47      0.45      1136

\nSaved:
  private_gold_overall_metrics.csv
  private_gold_per_label_metrics.csv
  private_gold_predictions.csv



# What to compare after the run

Record these four numbers:

1. **Base BanglaBERT OOF**
2. **BanglaBERT-large OOF**
3. **Best ensemble OOF**
4. **Kaggle public score**

Also send the printed per-label OOF/error analysis.

### If the score is still below 0.61

Do not immediately add more models.

The error-analysis CSV will tell us which labels are responsible. The likely next experiment would then be targeted at the weakest attribute(s), rather than increasing architecture complexity everywhere.
